# **Papers 13–15 — overlap, CVaR size, AAOIFI warning band**

Three diagnostics on the **locked live book** (FCF 100%, 10% name cap, SMA → cash, next-open AAOIFI exits). None of these is a new engine until the numbers say so.

**13. Overlap.** Do FCF, ROIC, dual-momentum, and SUE hold the same stocks? If they do, mixing them is not a second book.

**14. CVaR size.** Keep the same names. Give more weight to names whose worst 5% of days were milder, less weight to names whose worst days were worse. Compare to cap-weight.

**15. Warning band.** Debt ≥ 28%, cash ≥ 28%, or receivables ≥ 68% is close to the 30 / 30 / 70 fail line. Drop those names at the month-end they sit in the band. Compare to paper 10, which sells only after the name actually fails, at the next session.

**Frozen from 07–12:** `cost_bps=10` is not applied here (paper 12 already priced trades). Purification is not applied (paper 11). Kill bars: 2022 more than 5 points worse than SPUS, or train max drawdown worse than SPUS.

## Setup

In [1]:
import os
import subprocess
import sys
from pathlib import Path

from IPython.display import display


def resolve_notebook_dir() -> Path:
    candidates = [Path.cwd()]
    candidates.append(
        Path.home() / "Documents/Development/Monterey-Finance/Research/papers/13-15-diagnostics"
    )
    for path in candidates:
        try:
            resolved = path.resolve()
        except OSError:
            continue
        if resolved.is_dir() and (resolved / "code.ipynb").exists():
            return resolved
    return Path.cwd()


NB_DIR = resolve_notebook_dir()
os.chdir(NB_DIR)

REPO_ROOT = NB_DIR.parents[2]
RESEARCH_ROOT = NB_DIR.parents[1]
PAPER07 = RESEARCH_ROOT / "papers" / "07-book-construction"
PAPER10 = RESEARCH_ROOT / "papers" / "10-compliance-exits"
HQ_ROOT = REPO_ROOT.parent / "halalquant"
PIP_DEPS = ["matplotlib", "pyarrow", "duckdb"]

if HQ_ROOT.is_dir() and (HQ_ROOT / "pyproject.toml").exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f"{HQ_ROOT}[cache]"],
        cwd=str(NB_DIR),
        check=False,
    )
    if str(HQ_ROOT) not in sys.path:
        sys.path.insert(0, str(HQ_ROOT))
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + PIP_DEPS, cwd=str(NB_DIR), check=False)
if str(RESEARCH_ROOT) not in sys.path:
    sys.path.insert(0, str(RESEARCH_ROOT))

import importlib
import sleeves
import sleeves.backtest
import sleeves.book
import sleeves.diagnostics
import sleeves.exits
import sleeves.lab

importlib.reload(sleeves.diagnostics)
importlib.reload(sleeves.exits)
importlib.reload(sleeves.backtest)
importlib.reload(sleeves.book)
importlib.reload(sleeves.lab)
importlib.reload(sleeves)

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sleeves import FrozenRules, Lab, calc_performance_stats, run_sleeve
from sleeves.backtest import run_weight_schedule
from sleeves.diagnostics import (
    boundary_flags,
    drop_near_boundary,
    holdings_sets,
    jaccard,
    overlap_weight,
    reweight_log_cvar,
)
from sleeves.exits import apply_breach_exits, detect_breaches
from sleeves.prices import spy_sma_risk_on
from sleeves.purify import weights_from_log
from sleeves.rules import FrozenRules as FR
from sleeves.triggers import rebalance_dates
from sleeves.weighting import apply_name_cap

warnings.filterwarnings("ignore", category=FutureWarning)

START = "2019-12-19"
END = "2024-12-31"
TRAIN_START, TRAIN_END = "2020-01-01", "2022-12-31"
TEST_START, TEST_END = "2023-01-01", "2024-12-31"
RISK_FREE_RATE = 0.02
NAME_CAP = 0.10
ENGINE = {"fcf_quality": 1.0}
KILL_2022_GAP = 0.05
SLEEVE_IDS = ("fcf_quality", "roic", "dual_momentum", "sue")
SLEEVE_LABELS = {
    "fcf_quality": "FCF",
    "roic": "ROIC",
    "dual_momentum": "Dual mom",
    "sue": "SUE",
}

CACHE_DIR = PAPER07 / "cache"
FIG_DIR = NB_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"notebook dir: {NB_DIR}")
print("engine: FCF 100%  name_cap=10%  SMA-to-cash  next_open exits")


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


notebook dir: /Users/raoabdul/Documents/Development/Monterey-Finance/Research/papers/13-15-diagnostics
engine: FCF 100%  name_cap=10%  SMA-to-cash  next_open exits


## Step 1 — Locked book and standalone sleeves

In [2]:
metrics = pd.read_parquet(CACHE_DIR / "metrics.parquet")
prices = pd.read_parquet(CACHE_DIR / "prices.parquet")
prices["date"] = pd.to_datetime(prices["date"])
roic_history = pd.read_parquet(CACHE_DIR / "roic_history.parquet")
sue_events = pd.read_parquet(CACHE_DIR / "sue_events.parquet")
print(f"metrics {metrics.shape}  prices {prices.shape}  roic {roic_history.shape}  sue {sue_events.shape}")

rules = FrozenRules().with_book(
    sleeve_weights=ENGINE,
    name_cap=NAME_CAP,
    throttle="spy_sma",
    collapse_share_classes=True,
    risk_free_rate=RISK_FREE_RATE,
    rebalance_freq="ME",
    breach_exit="next_open",
    breach_monitor="filings",
    purify_schedule="ex_date",
)
lab = Lab.from_frames(
    metrics,
    prices,
    rules=rules,
    roic_history=roic_history,
    sue_events=sue_events,
    start=START,
    end=END,
)
print("Running locked book …", flush=True)
base_ret, base_log = lab.book().run(start=START)


def to_series(frame):
    s = frame.dropna(subset=["return"]).set_index("date")["return"].astype(float)
    s.index = pd.to_datetime(s.index)
    return s.sort_index()


live = to_series(base_ret)
panel = lab.price_panel()
spy_px = lab.spy_series()
window = lab.rules.dual_momentum.sma_window
throttle_on = pd.Series(
    [spy_sma_risk_on(spy_px, dt, window) for dt in panel.index],
    index=pd.to_datetime(panel.index),
)
events = detect_breaches(
    metrics, base_log, panel, rules=rules, monitor="filings", end=END, throttle_on=throttle_on
)
live_w = apply_breach_exits(
    weights_from_log(base_log), events, panel.index, lag="next_open", name_cap=NAME_CAP
)
print(f"live log rows {len(base_log)}  breach events {len(events)}  weight dates {len(live_w)}")

sleeve_rets = {}
sleeve_logs = {}
for sid in SLEEVE_IDS:
    print("Sleeve", sid, flush=True)
    r, lg = run_sleeve(lab, sid, start=START)
    sleeve_rets[sid] = to_series(r)
    sleeve_logs[sid] = lg
    print(f"  days {len(sleeve_rets[sid])}  log {len(lg)}")

bench_px = (
    prices[prices["symbol"].isin(["SPY", "SPUS"])]
    .pivot(index="date", columns="symbol", values="adj_close")
    .sort_index()
    .ffill()
)
bench_ret = bench_px.pct_change(fill_method=None)
spy = bench_ret["SPY"].dropna()
spus = bench_ret["SPUS"].dropna()
spy = spy[(spy.index >= pd.Timestamp(START)) & (spy.index <= pd.Timestamp(END))]
spus = spus[(spus.index >= pd.Timestamp(START)) & (spus.index <= pd.Timestamp(END))]

metrics (25600, 21)  prices (778360, 8)  roic (1724, 9)  sue (20106, 10)
Running locked book …


live log rows 6769  breach events 25  weight dates 81
Sleeve fcf_quality


  days 1259  log 6769
Sleeve roic


  days 524  log 1913
Sleeve dual_momentum


  days 1259  log 2429
Sleeve sue


  days 1259  log 16810


# Paper 13 — same names or different books?

On each month-end, take the stock list of each sleeve (dual-momentum list with the SMA **off the selector**, so cash months are not counted as empty). Jaccard is shared names ÷ union. Weighted overlap is the sum of min(weight A, weight B). Daily return correlation uses each sleeve’s own P&L, including dual-momentum cash when SPY is below the 200-day average.

In [3]:
list_rules = FrozenRules().with_book(
    sleeve_weights=ENGINE,
    name_cap=None,
    throttle="off",
    collapse_share_classes=True,
)
list_rules.dual_momentum.apply_sma = False
list_lab = Lab.from_frames(
    metrics,
    prices,
    rules=list_rules,
    roic_history=roic_history,
    sue_events=sue_events,
    start=START,
    end=END,
)

rebals = [d for d in rebalance_dates(metrics) if pd.Timestamp(d) >= pd.Timestamp(START)]
hold_by_date = {sid: {} for sid in SLEEVE_IDS}
w_by_date = {sid: {} for sid in SLEEVE_IDS}
for as_of in rebals:
    for sid in SLEEVE_IDS:
        h = list_lab.holdings(sid, as_of)
        if h is None or h.empty:
            hold_by_date[sid][as_of] = set()
            w_by_date[sid][as_of] = pd.Series(dtype=float)
            continue
        hold_by_date[sid][as_of] = set(h["symbol"].astype(str))
        w_by_date[sid][as_of] = h.set_index("symbol")["weight"].astype(float)

pairs = [(a, b) for i, a in enumerate(SLEEVE_IDS) for b in SLEEVE_IDS[i + 1 :]]
overlap_rows = []
jmat = pd.DataFrame(index=[SLEEVE_LABELS[s] for s in SLEEVE_IDS], columns=[SLEEVE_LABELS[s] for s in SLEEVE_IDS], dtype=float)
wmat = jmat.copy()
for a in SLEEVE_IDS:
    jmat.loc[SLEEVE_LABELS[a], SLEEVE_LABELS[a]] = 1.0
    wmat.loc[SLEEVE_LABELS[a], SLEEVE_LABELS[a]] = 1.0
for a, b in pairs:
    js, ws = [], []
    for d in rebals:
        sa, sb = hold_by_date[a][d], hold_by_date[b][d]
        if not sa and not sb:
            continue
        js.append(jaccard(sa, sb))
        ws.append(overlap_weight(w_by_date[a][d], w_by_date[b][d]))
    mean_j = float(np.nanmean(js)) if js else np.nan
    mean_w = float(np.nanmean(ws)) if ws else np.nan
    overlap_rows.append(
        {
            "pair": f"{SLEEVE_LABELS[a]} vs {SLEEVE_LABELS[b]}",
            "mean_jaccard": mean_j,
            "mean_weight_overlap": mean_w,
            "n_dates": len(js),
        }
    )
    jmat.loc[SLEEVE_LABELS[a], SLEEVE_LABELS[b]] = mean_j
    jmat.loc[SLEEVE_LABELS[b], SLEEVE_LABELS[a]] = mean_j
    wmat.loc[SLEEVE_LABELS[a], SLEEVE_LABELS[b]] = mean_w
    wmat.loc[SLEEVE_LABELS[b], SLEEVE_LABELS[a]] = mean_w

overlap = pd.DataFrame(overlap_rows)
display(overlap)
overlap.to_csv(NB_DIR / "sleeve_overlap.csv", index=False)

ret_df = pd.DataFrame({SLEEVE_LABELS[s]: sleeve_rets[s] for s in SLEEVE_IDS}).dropna(how="any")
corr = ret_df.corr()
display(corr)
corr.to_csv(NB_DIR / "sleeve_return_corr.csv")

# One date example: last rebalance with all four lists non-empty
example_date = None
for d in reversed(rebals):
    if all(hold_by_date[s][d] for s in SLEEVE_IDS):
        example_date = d
        break
if example_date is None:
    example_date = rebals[-1]
print(f"Example date {example_date}")
shared_fcf_roic = hold_by_date["fcf_quality"][example_date] & hold_by_date["roic"][example_date]
print(
    f"FCF n={len(hold_by_date['fcf_quality'][example_date])}  "
    f"ROIC n={len(hold_by_date['roic'][example_date])}  "
    f"shared {len(shared_fcf_roic)}"
)
print("shared sample:", sorted(shared_fcf_roic)[:15])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, mat, title in (
    (axes[0], jmat.astype(float), "Mean Jaccard (names)"),
    (axes[1], corr, "Daily return correlation"),
):
    im = ax.imshow(mat.values, vmin=0, vmax=1, cmap="Blues")
    ax.set_xticks(range(len(mat.columns)))
    ax.set_yticks(range(len(mat.index)))
    ax.set_xticklabels(mat.columns, rotation=30, ha="right")
    ax.set_yticklabels(mat.index)
    ax.set_title(title)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat.values[i, j]
            if pd.notna(val):
                ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()
fig.savefig(FIG_DIR / "overlap.png", dpi=140)
plt.show()

# Live-book concentration (top 5 weight on month-ends)
top5 = (
    base_log.groupby("as_of", group_keys=False)
    .apply(lambda g: float(g.nlargest(5, "weight")["weight"].sum()))
)
print(f"Live book mean top-5 weight {float(top5.mean()):.1%}")


,pair,mean_jaccard,mean_weight_overlap,n_dates
0,FCF vs ROIC,0.093674,0.437197,61
1,FCF vs Dual mom,0.162161,0.262584,61
2,FCF vs SUE,0.060620,0.057386,61
3,ROIC vs Dual mom,0.056255,0.298673,61
4,ROIC vs SUE,0.015371,0.023545,60
5,Dual mom vs SUE,0.041511,0.040730,61


,FCF,ROIC,Dual mom,SUE
FCF,1.000000,0.919693,0.776917,0.545789
ROIC,0.919693,1.000000,0.801336,0.516819
Dual mom,0.776917,0.801336,1.000000,0.360239
SUE,0.545789,0.516819,0.360239,1.000000


Example date 2024-11-30
FCF n=117  ROIC n=80  shared 42
shared sample: ['A', 'ADBE', 'ADSK', 'AKAM', 'ALGN', 'AMAT', 'ANET', 'CDNS', 'CF', 'CRM', 'CSGP', 'DVN', 'DXCM', 'FFIV', 'FTNT']
Live book mean top-5 weight 36.3%


/var/folders/lf/wzrn2w4j7mq_hf01km30xn0c0000gn/T/ipykernel_17249/830296465.py:106: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# Paper 14 — same names, CVaR weights

On each live-book date, replace cap-weights with 1 / |CVaR 5%| from the prior 60 trading days, then clip at 10%. Names with no history keep no weight that month. SMA cash and next-open exits stay as in the live book.

In [4]:
print("CVaR reweight …", flush=True)
cvar_month = reweight_log_cvar(base_log, panel, name_cap=NAME_CAP, lookback=60, alpha=0.05)
print(f"CVaR month-ends {len(cvar_month)}")

# Patch intra-month next_open dates: same names as live_w, CVaR sizes from that day's members
from sleeves.diagnostics import trailing_cvar_by_symbol, cvar_weights

cvar_w = {}
for as_of, w in live_w.items():
    names = list(w.index)
    cv = trailing_cvar_by_symbol(panel, as_of, lookback=60, alpha=0.05)
    nw = cvar_weights(names, cv, name_cap=NAME_CAP)
    cvar_w[as_of] = nw if not nw.empty else w

cvar_ret = run_weight_schedule(prices, cvar_w, start=START, throttle_on=throttle_on)
cvar_s = to_series(cvar_ret)
print(f"CVaR days {len(cvar_s)}")

# Example: last live month-end with both schemes
ex = max(d for d in cvar_month if d in weights_from_log(base_log))
cap_ex = weights_from_log(base_log)[ex].sort_values(ascending=False)
cvar_ex = cvar_month[ex].sort_values(ascending=False)
cmp = pd.DataFrame({"cap_weight": cap_ex, "cvar_weight": cvar_ex}).fillna(0.0)
cmp["shift"] = cmp["cvar_weight"] - cmp["cap_weight"]
print(f"Weight example {ex}  (largest CVaR cuts vs cap)")
display(cmp.sort_values("shift").head(8))
cmp.sort_values("cap_weight", ascending=False).head(15).to_csv(NB_DIR / "cvar_weight_example.csv")

CVaR reweight …


CVaR month-ends 61


CVaR days 1259


Weight example 2024-12-31  (largest CVaR cuts vs cap)


,cap_weight,cvar_weight,shift
symbol,,,
MSFT,0.100000,0.006716,-0.093284
GOOG,0.100000,0.008000,-0.092000
AAPL,0.100000,0.012898,-0.087102
META,0.091049,0.007239,-0.083809
AVGO,0.064260,0.005712,-0.058547
ORCL,0.027155,0.005812,-0.021343
ABBV,0.018636,0.004678,-0.013957
NVDA,0.019572,0.006226,-0.013346


# Paper 15 — drop names in the 28% / 28% / 68% band

Flag a holding if, on that month-end snapshot, debt/MC ≥ 0.28, cash/MC ≥ 0.28, or receivables/MC ≥ 0.68. Zero it out and renormalize. Then apply paper 10 next-open exits on what remains. Compare flags to `filing_breaches.csv`: was the name already in the band at the last rebalance, or did the new filing jump the line?

In [5]:
flags = boundary_flags(base_log, metrics)
print(f"warning-band name-months {len(flags)}  unique names {flags['symbol'].nunique() if not flags.empty else 0}")
if not flags.empty:
    display(flags.groupby("near").size().rename("n").to_frame())
    display(flags.sort_values("gap_to_limit").head(12))
flags.to_csv(NB_DIR / "boundary_flags.csv", index=False)

breaches = pd.read_csv(PAPER10 / "filing_breaches.csv")
breaches["rebalance"] = pd.to_datetime(breaches["rebalance"]).dt.date
flag_keys = set(zip(pd.to_datetime(flags["as_of"]).dt.date, flags["symbol"].astype(str))) if not flags.empty else set()
caught = []
for _, row in breaches.iterrows():
    key = (row["rebalance"], str(row["symbol"]))
    caught.append({**row.to_dict(), "flagged_at_rebalance": key in flag_keys})
caught_df = pd.DataFrame(caught)
n_catch = int(caught_df["flagged_at_rebalance"].sum()) if not caught_df.empty else 0
print(f"paper 10 filing breaches {len(breaches)}  already in warning band at last month-end {n_catch}")
caught_df.to_csv(NB_DIR / "breach_vs_boundary.csv", index=False)
display(caught_df[["symbol", "rebalance", "detect_date", "reason", "debt_ratio", "cash_ratio", "flagged_at_rebalance"]].head(15))

month_w = weights_from_log(base_log)
trimmed_month = drop_near_boundary(month_w, flags, name_cap=NAME_CAP)
trim_w = apply_breach_exits(
    trimmed_month, events, panel.index, lag="next_open", name_cap=NAME_CAP
)
trim_ret = run_weight_schedule(prices, trim_w, start=START, throttle_on=throttle_on)
trim_s = to_series(trim_ret)
print(f"trim days {len(trim_s)}")


warning-band name-months 179  unique names 41


,n
near,
cash,21
debt,158


,symbol,as_of,weight,debt_ratio,cash_ratio,receivables_ratio,near,gap_to_limit
122,EXPE,2023-03-31,0.001031,0.299999,0.199230,0.299133,debt,0.000001
176,URI,2024-12-31,0.002803,0.299867,0.009451,0.067508,debt,0.000133
60,ICE,2021-05-31,0.006227,0.299823,0.036098,0.058398,debt,0.000177
178,YUM,2024-12-31,0.002262,0.299818,0.013558,0.013558,debt,0.000182
108,FISV,2023-01-31,0.006786,0.299770,0.011786,0.052157,debt,0.000230
153,MMM,2024-02-29,0.003056,0.299727,0.112264,0.201052,debt,0.000273
128,EBAY,2023-04-30,0.001770,0.299612,0.161407,0.164447,debt,0.000388
81,IQV,2022-05-31,0.003832,0.299563,0.036767,0.100269,debt,0.000437
149,FISV,2024-01-31,0.005872,0.299217,0.012601,0.062685,debt,0.000783
106,FISV,2022-12-31,0.006590,0.299058,0.011758,0.052033,debt,0.000942


paper 10 filing breaches 25  already in warning band at last month-end 7


,symbol,rebalance,detect_date,reason,debt_ratio,cash_ratio,flagged_at_rebalance
0,HLT,2020-01-31,2020-02-11,debt ratio exceeds threshold,0.313013,0.021167,True
1,FIS,2020-01-31,2020-02-20,debt ratio exceeds threshold,0.523108,0.030053,False
2,NOW,2020-01-31,2020-02-20,cash ratio exceeds threshold,0.000000,0.338165,False
3,ABBV,2020-01-31,2020-02-21,debt ratio exceeds threshold,0.465506,0.295115,True
4,GPN,2020-01-31,2020-02-21,debt ratio exceeds threshold,0.421296,0.077780,False
5,BMY,2020-01-31,2020-02-24,debt ratio exceeds threshold,0.527442,0.173730,False
6,AMT,2020-01-31,2020-02-25,debt ratio exceeds threshold,0.304518,0.019004,False
7,FISV,2020-01-31,2020-02-27,debt ratio exceeds threshold,0.608661,0.000000,False
8,LITE,2020-07-31,2020-08-25,debt ratio exceeds threshold,0.324839,0.328048,False
9,ICE,2021-01-31,2021-02-04,debt ratio exceeds threshold,0.322819,0.038866,False


trim days 1259


## Scorecard vs SPUS

In [6]:
def sl(s, a, b):
    return s[(s.index >= pd.Timestamp(a)) & (s.index <= pd.Timestamp(b))].dropna()


def calendar_return(s, year):
    x = sl(s, f"{year}-01-01", f"{year}-12-31")
    return float((1 + x).prod() - 1) if not x.empty else np.nan


def score(s, label):
    full = sl(s, START, END)
    train = sl(s, TRAIN_START, TRAIN_END)
    test = sl(s, TEST_START, TEST_END)
    st = calc_performance_stats(full, label, rf=RISK_FREE_RATE)
    st_tr = calc_performance_stats(train, label, rf=RISK_FREE_RATE)
    st_te = calc_performance_stats(test, label, rf=RISK_FREE_RATE)
    return {
        "book": label,
        "ret_2020": calendar_return(s, 2020),
        "ret_2022": calendar_return(s, 2022),
        "ret_2023": calendar_return(s, 2023),
        "ret_2024": calendar_return(s, 2024),
        "full_cagr": st["CAGR"],
        "full_vol": st["Volatility"],
        "full_sharpe": st["Sharpe"],
        "full_max_dd": st["Max Drawdown"],
        "full_calmar": st["Calmar"],
        "train_max_dd": st_tr["Max Drawdown"],
        "test_cagr": st_te["CAGR"],
    }


series_map = {
    "Live (cap-weight)": live,
    "CVaR 5% weights": cvar_s,
    "Trim warning band": trim_s,
    "S&P 500": spy,
    "Halal (SPUS)": spus,
}
for sid in SLEEVE_IDS:
    series_map[f"Sleeve {SLEEVE_LABELS[sid]}"] = sleeve_rets[sid]

table = pd.DataFrame([score(s, k) for k, s in series_map.items()])
spus_row = table.loc[table["book"] == "Halal (SPUS)"].iloc[0]


def kill_reason(row):
    if row["book"] in {"S&P 500", "Halal (SPUS)"} or str(row["book"]).startswith("Sleeve "):
        return ""
    reasons = []
    if pd.notna(row["ret_2022"]) and row["ret_2022"] < spus_row["ret_2022"] - KILL_2022_GAP:
        reasons.append("2022 >5ppt worse than SPUS")
    if pd.notna(row["train_max_dd"]) and row["train_max_dd"] < spus_row["train_max_dd"]:
        reasons.append("train max DD worse than SPUS")
    return "; ".join(reasons)


table["kill"] = table.apply(kill_reason, axis=1)
table["pass"] = table["kill"] == ""
show = table.copy()
for c in ("ret_2020", "ret_2022", "ret_2023", "ret_2024", "full_cagr", "full_vol", "full_max_dd", "train_max_dd", "test_cagr"):
    show[c] = show[c].map(lambda x: "" if pd.isna(x) else f"{x:.2%}")
for c in ("full_sharpe", "full_calmar"):
    show[c] = show[c].map(lambda x: "" if pd.isna(x) else f"{x:.2f}")
display(show)
table.to_csv(NB_DIR / "diagnostics_scorecard.csv", index=False)

fig, ax = plt.subplots(figsize=(11, 5))
for k in ("Live (cap-weight)", "CVaR 5% weights", "Trim warning band", "Halal (SPUS)"):
    eq = (1 + sl(series_map[k], START, END)).cumprod()
    ax.plot(eq.index, eq.values, label=k)
ax.set_title("Growth of $1 — live book vs CVaR size vs warning-band trim")
ax.set_ylabel("Growth of $1")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "equity-curves.png", dpi=140)
plt.show()

,book,ret_2020,ret_2022,ret_2023,ret_2024,full_cagr,full_vol,full_sharpe,full_max_dd,full_calmar,train_max_dd,test_cagr,kill,pass
0,Live (cap-weight),32.28%,0.97%,35.61%,20.85%,23.34%,13.56%,1.47,-9.04%,2.58,-9.04%,28.15%,,True
1,CVaR 5% weights,39.52%,1.90%,24.23%,13.26%,20.97%,12.45%,1.43,-9.37%,2.24,-9.37%,18.70%,,True
2,Trim warning band,32.77%,1.14%,35.83%,20.39%,23.40%,13.61%,1.47,-9.04%,2.59,-9.04%,28.00%,,True
3,S&P 500,18.33%,-18.18%,26.18%,25.34%,14.75%,20.94%,0.67,-33.72%,0.44,-33.72%,25.93%,,True
4,Halal (SPUS),25.68%,-22.76%,34.20%,27.67%,17.78%,21.58%,0.77,-30.80%,0.58,-30.80%,31.11%,,True
5,Sleeve FCF,24.94%,-22.93%,30.25%,20.85%,14.59%,21.62%,0.65,-29.49%,0.49,-29.49%,25.58%,,True
6,Sleeve ROIC,,-8.20%,37.19%,24.63%,24.21%,16.19%,1.30,-11.98%,2.02,-11.98%,30.90%,,True
7,Sleeve Dual mom,11.91%,-24.04%,4.10%,19.46%,5.40%,17.47%,0.27,-30.22%,0.18,-26.19%,11.56%,,True
8,Sleeve SUE,32.99%,-2.31%,60.44%,59.42%,35.23%,24.52%,1.27,-39.18%,0.90,-39.18%,60.23%,,True


/var/folders/lf/wzrn2w4j7mq_hf01km30xn0c0000gn/T/ipykernel_17249/3135364824.py:77: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Picks

In [7]:
live_row = table.loc[table["book"] == "Live (cap-weight)"].iloc[0]
cvar_row = table.loc[table["book"] == "CVaR 5% weights"].iloc[0]
trim_row = table.loc[table["book"] == "Trim warning band"].iloc[0]

print("Overlap (paper 13)")
print(overlap.to_string(index=False))
print()
print("Return correlation")
print(corr.round(2).to_string())
print()

fcf_roic_j = float(overlap.loc[overlap["pair"] == "FCF vs ROIC", "mean_jaccard"].iloc[0]) if not overlap.empty else np.nan
fcf_roic_w = float(overlap.loc[overlap["pair"] == "FCF vs ROIC", "mean_weight_overlap"].iloc[0]) if not overlap.empty else np.nan
fcf_roic_c = float(corr.loc["FCF", "ROIC"]) if "FCF" in corr.index else np.nan
# Name Jaccard can be low when one list is long. Dollars and daily P&L are the blend test.
if (pd.notna(fcf_roic_w) and fcf_roic_w >= 0.30) or (pd.notna(fcf_roic_c) and fcf_roic_c >= 0.80):
    pick13 = "keep one engine (FCF). Do not add ROIC / dual-mom / SUE as extra sleeves."
else:
    pick13 = "lists differ enough to re-open a blend test; still require 2020–2022 path."
print(f"PICK 13: {pick13}")
print(f"  FCF vs ROIC Jaccard {fcf_roic_j:.1%}  weight overlap {fcf_roic_w:.1%}  daily corr {fcf_roic_c:.2f}")

dd_help = (cvar_row["full_max_dd"] - live_row["full_max_dd"]) > 0.005  # less negative
cagr_ok = cvar_row["full_cagr"] >= live_row["full_cagr"] - 0.01
if bool(cvar_row["pass"]) and dd_help and cagr_ok:
    pick14 = "switch live size rule to CVaR 5% / 60-day, 10% cap."
else:
    pick14 = "keep cap-weight. CVaR is a report, not the live size rule."
print(f"PICK 14: {pick14}")
print(
    f"  live CAGR {live_row['full_cagr']:.2%} DD {live_row['full_max_dd']:.2%}  "
    f"CVaR CAGR {cvar_row['full_cagr']:.2%} DD {cvar_row['full_max_dd']:.2%}  pass={cvar_row['pass']}"
)

if n_catch / max(len(breaches), 1) >= 0.5 and bool(trim_row["pass"]) and (trim_row["full_max_dd"] - live_row["full_max_dd"]) > 0.002:
    pick15 = "drop warning-band names at month-end (on top of next_open fails)."
else:
    pick15 = "keep next_open only. Warning band is a watch list, not a sell rule."
print(f"PICK 15: {pick15}")
print(f"  filing breaches already in band {n_catch}/{len(breaches)}")
print(
    f"  trim CAGR {trim_row['full_cagr']:.2%} DD {trim_row['full_max_dd']:.2%}  pass={trim_row['pass']}"
)
print()
print("Live rule still:")
print("  FrozenRules().with_book(")
print("      sleeve_weights={'fcf_quality': 1.0},")
print("      name_cap=0.10,")
print("      throttle='spy_sma',")
print("      breach_exit='next_open',")
print("      purify_schedule='ex_date',")
print("      cost_bps=10,")
print("  )")


Overlap (paper 13)
            pair  mean_jaccard  mean_weight_overlap  n_dates
     FCF vs ROIC      0.093674             0.437197       61
 FCF vs Dual mom      0.162161             0.262584       61
      FCF vs SUE      0.060620             0.057386       61
ROIC vs Dual mom      0.056255             0.298673       61
     ROIC vs SUE      0.015371             0.023545       60
 Dual mom vs SUE      0.041511             0.040730       61

Return correlation
           FCF  ROIC  Dual mom   SUE
FCF       1.00  0.92      0.78  0.55
ROIC      0.92  1.00      0.80  0.52
Dual mom  0.78  0.80      1.00  0.36
SUE       0.55  0.52      0.36  1.00

PICK 13: keep one engine (FCF). Do not add ROIC / dual-mom / SUE as extra sleeves.
  FCF vs ROIC Jaccard 9.4%  weight overlap 43.7%  daily corr 0.92
PICK 14: keep cap-weight. CVaR is a report, not the live size rule.
  live CAGR 23.34% DD -9.04%  CVaR CAGR 20.97% DD -9.37%  pass=True
PICK 15: keep next_open only. Warning band is a watch list, not

## How to read the picks

- Paper 13 is about names, not returns. High Jaccard means two ranking rules pick the same tickers.
- Paper 14 does not change which names you own. It only changes how large each line is.
- Paper 15 sells *before* a fail. Paper 10 sells *after* a fail. A filing that jumps from 20% debt to 47% debt will not be in the 28% band the month before.
- None of these three replace the SMA cash switch or the 10% name cap.